TENSORI: OPERAZIONI BASE IN TENSORFLOW

Passiamo da un'officina meccanica dove tutto è fatto a mano, ad una fabbrica ad alta precisione.
Non sono strumenti opposti ma filosofie diverse.

- Entità immutabili e dinamiche
- Il potere della vettorizzazione
- Algebra lineare applicata.

Per far imparare una rete neurale, dobbiamo definire dei confini rigidi dentro i quali avviene il cambiamento. Immagina di allestrire un teatro, il palco e le luci sono il perimetro fisso, mentre gli attori cambiano posizione. In TensorFlow separare ciò che è statico da ciò che è in movimento non è un vezzo estetico ma una necessità computazionale. Serve a dire alla GPU, questo non toccarlo, tienilo pronto nei registri veloci, questo invece traccialo perchè sta per cambiare.
Il framework ha bisogno di sapere in anticipo quali dasti rimarrano fissi, come gli iperparametri o gli input, e quali dovranno evolversi durante l'addestramento.
Questa situazione non solo è sintattica, ma permette a TensoFlow di ottimizzare l'allocazione della memoria sulla GPU e di tracciare correttamente i cambiamenti per il calcolo dei gradienti.

Differenze strutturali trai icontenitori
- tf.constant: crea tensori immutabili. Una volta definiti, i loro valori risiedono in una porzione di memoria protetta e non possono essere alterati. Perfetta per i dati di input ed i dati che si decide di non modificare.
- tf.Variable: rappresentano lo stato del modello. E' l'unico oggetto che permette modifice in-place, rendendolo perfetto per ospitare pesi (W) e bias (b).
- Dtype: a differenza delle liste Python, TensoFlow richiede che ogni elemento del tensore abbia lo stesso tipo (es.float32), fondamentale per il calolo parallelo.
- Eager Execution: di default, TensorFlow 2.x esegue le operazioni immediatamente, permettendo di ispezionare i valori senza dover inizializzare una sessione esplicita.

Manipolazione delle Variabili
- Metodo assign: per modificare una ts.Variable non si usa l'operatore uguale standard Python, ma il metodo .assign(), che aggiorna il valore preservando il riferimento nel grafo. TensorFlow non sta solo cambiando un numero, sta aggiornando un nodo in un grafo complesso. Come se, invece di cancellare un numero sulla lavana, mandassimo un messaggio a tutta la rete del tipo aggiornate i vostri calcoli, il peso è cambiato.
- Operazioni cumulative: Funzioni come .assign_add() o .assign_sub() sono essenziali durante il Gradient Descent per sommare o sottrarre i gradienti calcolati ai pesi attuali.
- Interoperabilità Numpy: TensorFlow converte automaticamente gli array Numpy in costanti, ,ma è buona pratica definirli esplicitamente per controllare la precisione numerica.

Cicloi di Vita di una Variabile
Le variabili sono oggetti speciali che TensorFlow 'osserva' costantemente. TensorFlow ha un ispettore sempre attivo, chiamato tf.GradientTape, questo registra ogni minimo movimento della nostra ts.Variabile, se provassimo ad utilizzare una costante per aggiornare un peso il registratore non avrebbe nulla da registrare. Senza questo tracciamento non potremmo calcolare i gradienti.

Parallelismo Massivo
La forza del Deep Learning risiede nella capacità di eseguire calcoli su milioni di numeri contemporaneamente. Invece di usare cicli for, utilizziamo operazioni vettoriali (element-wise).
Diamo alla GPU un intero blocco di dati ed una singola operazione, il parallelismo esegue tutto. Questo grazie anche al fatto che i dati hanno la stessa forma


Calcolo distribuito
- Element-wise: operazioni come tf.add o tf.multiply applicano la funzione a ogni singola coppia di elementi corrispondenti nei tensori di input
- Broadcasting: meccanismo che permette di operare su tensori di forme diverse, espandendo virtualmente il più piccolo per farlo coincidere con il più grande.
- Math Ops: TensorFlow offre un intero ecosistema di funzioni matematiche come tf.exp, ts.sql e tf.abs ottimizzare per GPU
- Reduzioni:operazioni come tf.reduce_mean aggregano i dati lungo un asse, trasformando ad esempio una matrice di errori in un singolo valore scalare di loss.


Brodcasting
Il broadcasting è come un'orchestra sincronizzata, se aggiungiamo un vettore ad una matrice TensorFlow non duplica i dati fisicamente, che saturerebbe la memoria della GPU, ma adatta virtualmente le dimensioni.
- Regole di compatibilità: Il broadcasting avviene se le dimensioni dei tensori sono uguali o se una di esse è pari a uno. Questo evita duplicazioni inutili di dati in memoria.
- Vantaggi prestazionali: lavorare in modo vettoriale permette ai core della GPU di eseguire la stessa istruzione su dati diversi (SIMD) abbattendo i tempi di calcolo.
- Operatori sovraccaricati: TensorFlow permette l'uso di +, -, * direttamente tra in tensori, traducendoli internamente nelle rispettive funzioni ottimizzate del framework.

Aggregazione Statistica
Il concetto di riduzione
In statistica e nel Deep Learning spesso dobbiamo passare dal dettaglio alla sintesi. La funzione reduce sono i nostri aggregatori
Molti algoritmi richiedono di sommare tutti gli elementi di un tensore o di trovarne la media. In TensorFlow usiamo il prefisso reduce_ per indicare che la dimensione del tensore risultante sarà minore dell'originale.
Specificando il parametro axis possiamo decidere se ridurre l'intero tensore o operare solo per righe o per colonne

Moltiplicazioni tra Matrici
Se il Deep Learning fosse il corpo umano, la tf.matmul sarebbe il cuore che pompa sangue. Ogni volta che un segnale passa da uno strato all'altro della rete, avviene una trasformazione geometrica, non stiamo solo moltiplicando i numeri, stiamo proiettando i dati da uno spazio all'altro, cercando di estrarre le caratteristiche salienti.

L'algebra di tf.matmul
- Compatibilità delle dimensioni: per moltiplicare due matrici (n*m) e (m*p), la dimensione interna m deve coincidere perfettamente. La larghezza della prima matrice deve corrispondere all'altezza della seconda.
- Non commutatività: l'ordine dei fattori è vitale. Moltiplicare l'input per i pesi (X *W) produce un risultato diverso rispetto a (W * X)
- Dot Product: ogni elemento della matrice risultante è la somma dei prodotti tra una riga della prima e una colonna della seconda
- Operatore @: TensoFlow supporta l'operato Python standard @ come scorciatoia sintattica per la funzione tf.matmul

Se i dati non sono orientati nel modo giusto, tf.matmul ci permette di trasporre i fattori al volo
Trasposizione e Ottimizzazione
- Parametri di tf.matmul: La funzione tf.matmul accetta argomenti come transose_a o transpose_b, permettendo di ruotare le matrici durante il calcolo senza creare copie itermedie
- Batch Matmul: Se i tensori hanno più di due dimensioni, TensorFlow moltiplica automaticamente le ultime due matrici per ogni elemento delle dimensioni iniziali (batch)
- Efficienza nei Tensor Core: Le moderne GPU hanno core dedicati esclusivamente a queste operazioni, rendendo la moltiplicazione tra metrici migliaia di volte più veloce di un ciclo for equivalente.

Dallo scalare al Tensore
Il passaggio al forward pass
Quando scriviamo un layer lineare, stiamo essenzialmente chiedendo a TensorFlow di prendere un vettore in input (X), moltiplicarlo per una matrice di pesi (W) e aggiungere un bias (b).
y=activation(x*W+b)
Capire tf.matmul significa capire come l'informazione viene compressa espansa o trasformata mentre attraversa la rete neurale.

In [2]:
import tensorflow as tf
import numpy as np

In [ ]:
#1. COSTANTI E VARIABILI: IL CUORE DELLO STATO
#Le costanti sono immutabili (input, iperparametri)
a=tf.constant([[1.0,2.0],[3.0,4.0]])

#Le variabili sono mutabili (pesi, bias)
#senza tf.variable, l'ottimizzatore non saprebbe cosa aggiornare
w=tf.Variable(tf.random.normal([2,1]),name="weights")
b=tf.Variable(tf.zeros([1]),name="bias")
print(f"pesi(W):\n\t{w}\nbias(b):\n\t{b}")

pesi(W):
	<tf.Variable 'weights:0' shape=(2, 1) dtype=float32, numpy=
array([[-0.11760399],
       [ 0.14054346]], dtype=float32)>
bias(b):
	<tf.Variable 'bias:0' shape=(1,) dtype=float32, numpy=array([0.], dtype=float32)>


In [15]:
#Modifica di una variabile: non usiamo =, usiamo .assign()
w.assign(w*2.0)
print(f"Variabile dopo raddoppio: {w}")

Variabile dopo raddoppio: <tf.Variable 'weights:0' shape=(2, 1) dtype=float32, numpy=
array([[-0.23520799],
       [ 0.28108692]], dtype=float32)>


In [ ]:
#2. OPERAZIONI VETTORIALI E BROADCASTING
#Calcolo element-wise accelerato
x=tf.constant([10,20,30],dtype=tf.float32) #variabili con valori costanti perchè sono falori in ingresso
y=tf.constant([1,2,3],dtype=tf.float32)

#somma
somma=tf.add(x,y) #Oppure x+y
#moltiplicazione
prodottoscalare=x*y
print(f"x={x}, y={y}")
print(f"Somma (x+y): {somma}")
print(f"Moltiplicazione (x*y): {prodottoscalare}")
print(f"Broadcasting (x*5): {x*5}") #broadcasting il 5 viene espanso a [5,5,5]


x=[10. 20. 30.], y=[1. 2. 3.]
Somma (x+y): [11. 22. 33.]
Moltiplicazione (x*y): [10. 40. 90.]
Broadcasting (x*5): [ 50. 100. 150.]


In [ ]:
#3. IL MOTORE DELLE ANN: TF.MATMUL
#Creiamo un piccolo batch di input (3 esempi, 2 feature ciascuno)
inputs=tf.constant([[1.0,2.0],[3.0,4.0],[5.0,6.6]])

#Moltiplicazione matriciale (3x2)@(2x1)=(3x1))
#Questo è il calcolo z=xW+b
z=tf.matmul(inputs,w)+b
print(f"inputs: {inputs}\tpesi(w):{w}\tbias(b):{b}")
print(f"Risultato Forward Pass (z=(inputs*w+b)):\n{z}")

inputs: [[1.  2. ]
 [3.  4. ]
 [5.  6.6]]	pesi(w):<tf.Variable 'weights:0' shape=(2, 1) dtype=float32, numpy=
array([[-0.11760399],
       [ 0.14054346]], dtype=float32)>	bias(b):<tf.Variable 'bias:0' shape=(1,) dtype=float32, numpy=array([0.], dtype=float32)>
Risultato Forward Pass (z=(inputs*w+b)):
[[0.16348293]
 [0.20936185]
 [0.3395669 ]]
